# NWSL 2026 Team Overview

Per-team performance table for the 2026 season.
Columns: xGF, xGA, xGD, xGD/g, GD-xGD, Pts, xPts, xPts/g, Pts-xPts.

Data: American Soccer Analysis.


In [1]:
from itscalledsoccer.client import AmericanSoccerAnalysis
import pandas as pd

from sophso import decorate_team_ids, nwsl_primary_colors

asa_client = AmericanSoccerAnalysis()

SEASON = "2026"

In [2]:
teams_df = decorate_team_ids(
    asa_client.get_team_xgoals(
        leagues="nwsl",
        season_name=SEASON,
        stage_name="Regular Season",
    )
)

# teams_df.columns.tolist()

In [3]:
overview_df = (
    teams_df.assign(
        gf_per_game=lambda d: d.goals_for / d.count_games,
        ga_per_game=lambda d: d.goals_against / d.count_games,
        gd_per_game=lambda d: d.goal_difference / d.count_games,
        xgd_per_game=lambda d: d.xgoal_difference / d.count_games,
        xpts_per_game=lambda d: d.xpoints / d.count_games,
        points_minus_xpoints=lambda d: d.points - d.xpoints,
    )[
        [
            "team",
            "count_games",
            "gf_per_game",
            "ga_per_game",
            "gd_per_game",
            "xgoals_for",
            "xgoals_against",
            "xgoal_difference",
            "xgd_per_game",
            "goal_difference_minus_xgoal_difference",
            "points",
            "xpoints",
            "xpts_per_game",
            "points_minus_xpoints",
        ]
    ]
    .rename(
        columns={
            "count_games": "GP",
            "gf_per_game": "GF/g",
            "ga_per_game": "GA/g",
            "gd_per_game": "GD/g",
            "xgoals_for": "xGF",
            "xgoals_against": "xGA",
            "xgoal_difference": "xGD",
            "xgd_per_game": "xGD/g",
            "goal_difference_minus_xgoal_difference": "GD-xGD",
            "points": "Pts",
            "xpoints": "xPts",
            "xpts_per_game": "xPts/g",
            "points_minus_xpoints": "Pts-xPts",
        }
    )
    .sort_values("xPts/g", ascending=False, ignore_index=True)
)

overview_df.round(2)

,team,GP,GF/g,GA/g,GD/g,xGF,xGA,xGD,xGD/g,GD-xGD,Pts,xPts,xPts/g,Pts-xPts
0,SD,20,1.20,1.05,0.15,32.23,17.62,14.61,0.73,-11.61,33,37.40,1.87,-4.40
1,NC,19,1.79,1.53,0.26,35.64,25.01,10.62,0.56,-5.62,30,32.70,1.72,-2.70
2,NJY,20,1.25,0.65,0.60,31.80,21.65,10.15,0.51,1.85,43,34.35,1.72,8.65
3,KC,20,1.50,1.40,0.10,35.36,25.86,9.50,0.48,-7.50,30,32.64,1.63,-2.64
4,WAS,20,1.45,0.85,0.60,29.93,22.31,7.62,0.38,4.38,37,32.50,1.62,4.50
5,UTA,20,1.70,1.15,0.55,32.43,26.31,6.12,0.31,4.88,36,32.21,1.61,3.79
6,LA,18,1.50,1.11,0.39,27.29,23.53,3.77,0.21,3.23,25,26.90,1.49,-1.90
7,ORL,20,1.20,1.55,-0.35,28.30,25.90,2.40,0.12,-9.40,23,29.37,1.47,-6.37
8,SEA,19,1.21,1.32,-0.11,26.89,27.34,-0.45,-0.02,-1.55,27,26.45,1.39,0.55
9,DEN,19,1.32,1.26,0.05,26.02,28.84,-2.82,-0.15,3.82,24,25.33,1.33,-1.33


## Strength of schedule via Elo

Use the xG+score Elo model from `elo-over-time_2026.ipynb` to estimate each
team's strength rating, then summarize each team's schedule:

- **Past Opp Str**: average pre-match strength of opponents in completed games
- **Future Opp Str**: average current strength of opponents in remaining games
- **Next 5 Opp Str**: average current strength of opponents in the next 5 scheduled games

Opponent strength includes a `home_points` adjustment when the opponent
plays at home, so the value reflects the opponent's effective strength on
match day rather than their raw rating.


In [4]:
# Elo helpers (kept in sync with elo-over-time_2026.ipynb)


def get_games(leagues="nwsl", seasons="2026", status=["FullTime", "PreMatch"]):
    games_df = decorate_team_ids(
        asa_client.get_games(leagues=leagues, seasons=seasons, status=status)
    )
    gxg_df = asa_client.get_game_xgoals(game_ids=list(games_df.game_id))

    games_df = games_df.merge(
        gxg_df[
            [
                "game_id",
                "home_team_xgoals",
                "away_team_xgoals",
                "home_xpoints",
                "away_xpoints",
            ]
        ],
        on="game_id",
        how="left",
    )
    games_df["date_time_utc"] = pd.to_datetime(games_df["date_time_utc"])
    games_df = games_df.sort_values("date_time_utc", ignore_index=True)
    return games_df


def calc_points_change(goal_diff, k, expected_result, goal_diff_draw=0):
    if abs(goal_diff) <= 1:
        goal_index = 1
    elif abs(goal_diff) < 3:
        goal_index = (1 + abs(goal_diff)) / 2
    else:
        goal_index = (11 + abs(goal_diff)) / 8

    if abs(goal_diff) <= goal_diff_draw:
        result = 0.5
    else:
        result = max(goal_diff / abs(goal_diff), 0)

    return k * goal_index * (result - expected_result)


def elo_learn(
    games_df,
    k=6,
    home_points=11,
    scaling_factor=100.0,
    learn_input="score",
    goal_diff_draw=0.3,
    rating_start=100.0,
):
    games_df = games_df.copy()

    if not "home_rating_pre" in games_df.columns:
        games_df["home_rating_pre"] = rating_start
        games_df["away_rating_pre"] = rating_start
        games_df["home_rating_post"] = rating_start
        games_df["away_rating_post"] = rating_start

    for gid in (
        games_df.sort_values("date_time_utc").query("status=='FullTime'").game_id
    ):
        game_dict = games_df.query(f"game_id == '{gid}'").to_dict(orient="records")[0]

        rating_diff = (
            game_dict["home_rating_pre"] - game_dict["away_rating_pre"] + home_points
        )
        expected_result = 1 / (1 + 10 ** (-rating_diff / scaling_factor))

        if learn_input == "score":
            points_change = calc_points_change(
                goal_diff=(game_dict["home_score"] - game_dict["away_score"]),
                k=k,
                expected_result=expected_result,
                goal_diff_draw=0,
            )
        elif learn_input == "xg":
            points_change = calc_points_change(
                goal_diff=(
                    game_dict["home_team_xgoals"] - game_dict["away_team_xgoals"]
                ),
                k=k,
                expected_result=expected_result,
                goal_diff_draw=goal_diff_draw,
            )
        elif ("xg" in learn_input) and ("score" in learn_input):
            points_change = calc_points_change(
                goal_diff=(game_dict["home_score"] - game_dict["away_score"]),
                k=k,
                expected_result=expected_result,
                goal_diff_draw=0,
            ) + calc_points_change(
                goal_diff=(
                    game_dict["home_team_xgoals"] - game_dict["away_team_xgoals"]
                ),
                k=k,
                expected_result=expected_result,
                goal_diff_draw=goal_diff_draw,
            )
        else:
            raise ("learn_input must be either 'score' or 'xg'")

        game_dict["home_rating_post"] = game_dict["home_rating_pre"] + points_change
        game_dict["away_rating_post"] = game_dict["away_rating_pre"] - points_change

        games_df.loc[games_df.game_id == gid, "home_rating_post"] = game_dict[
            "home_rating_post"
        ]
        games_df.loc[games_df.game_id == gid, "away_rating_post"] = game_dict[
            "away_rating_post"
        ]

        after_mask = games_df.date_time_utc > game_dict["date_time_utc"]

        games_df.loc[
            after_mask & (games_df.home_team == game_dict["home_team"]),
            "home_rating_pre",
        ] = game_dict["home_rating_post"]
        games_df.loc[
            after_mask & (games_df.away_team == game_dict["home_team"]),
            "away_rating_pre",
        ] = game_dict["home_rating_post"]
        games_df.loc[
            after_mask & (games_df.home_team == game_dict["away_team"]),
            "home_rating_pre",
        ] = game_dict["away_rating_post"]
        games_df.loc[
            after_mask & (games_df.away_team == game_dict["away_team"]),
            "away_rating_pre",
        ] = game_dict["away_rating_post"]

    return games_df

In [5]:
params = dict(
    home_points=10,
    k=6,
    scaling_factor=100,
    learn_input="xg+score",
    goal_diff_draw=0.3,
)

season_games_df = get_games(leagues="nwsl", seasons=SEASON)
elo_games_df = elo_learn(season_games_df, **params)

elo_games_df.status.value_counts()

status
FullTime    158
PreMatch     83
Name: count, dtype: int64

In [6]:
# Long-format: one row per (team, game) carrying opponent's pre-match strength.
# Opponent strength includes the home_points adjustment when opponent is at home.

home_view = elo_games_df.assign(
    team=elo_games_df.home_team,
    opponent=elo_games_df.away_team,
    opponent_rating_pre=elo_games_df.away_rating_pre,
    opponent_is_home=False,
)
away_view = elo_games_df.assign(
    team=elo_games_df.away_team,
    opponent=elo_games_df.home_team,
    opponent_rating_pre=elo_games_df.home_rating_pre,
    opponent_is_home=True,
)

team_games_df = pd.concat([home_view, away_view], ignore_index=True)[
    [
        "team",
        "opponent",
        "opponent_rating_pre",
        "opponent_is_home",
        "date_time_utc",
        "status",
        "matchday",
        "game_id",
    ]
].sort_values(["team", "date_time_utc"], ignore_index=True)

team_games_df["opponent_strength"] = (
    team_games_df["opponent_rating_pre"]
    + team_games_df["opponent_is_home"].astype(int) * params["home_points"]
)

team_games_df.head()

,team,opponent,opponent_rating_pre,opponent_is_home,date_time_utc,status,matchday,game_id,opponent_strength
0,BAY,DEN,100.000000,False,2026-03-14 22:30:00+00:00,FullTime,1,KPqj3vyYQ6,100.000000
1,BAY,LA,107.636374,False,2026-03-22 00:45:00+00:00,FullTime,2,e7MzynNGQr,107.636374
2,BAY,NC,100.296315,True,2026-03-28 23:00:00+00:00,FullTime,3,vzqoYv1JMa,110.296315
3,BAY,WAS,105.283573,False,2026-04-05 21:00:00+00:00,FullTime,4,eV5D8lnn5K,105.283573
4,BAY,NJY,90.853108,True,2026-04-25 17:00:00+00:00,FullTime,5,OlMlWv1DQL,100.853108


In [7]:
from matplotlib.colors import LinearSegmentedColormap


def opp_strength_summary(df):
    past = df.query("status == 'FullTime'")["opponent_strength"].mean()
    future_df = df.query("status == 'PreMatch'").sort_values("date_time_utc")
    return pd.Series(
        {
            "Past Opp Str": past,
            "Future Opp Str": future_df["opponent_strength"].mean(),
            "Next 5 Opp Str": future_df.head(5)["opponent_strength"].mean(),
        }
    )


strength_df = team_games_df.groupby("team").apply(opp_strength_summary).reset_index()

# Look up team short names so the table reads well; key colors by short name
# since the column values are about to be remapped from abbreviations.
teams_meta = asa_client.get_teams(leagues="nwsl")
short_name_map = dict(zip(teams_meta.team_abbreviation, teams_meta.team_short_name))
short_name_color_map = {
    short_name_map[abbr]: color
    for abbr, color in nwsl_primary_colors.items()
    if abbr in short_name_map
}

display_df = (
    overview_df.merge(strength_df, on="team", how="left")
    .assign(
        **{
            "xGF/g": lambda d: d["xGF"] / d["GP"],
            "xGA/g": lambda d: d["xGA"] / d["GP"],
        }
    )
    .drop(columns=["xGF", "xGA", "xGD"])[
        [
            "team",
            "GP",
            "GF/g",
            "GA/g",
            "GD/g",
            "xGF/g",
            "xGA/g",
            "xGD/g",
            "GD-xGD",
            "xPts/g",
            "Pts",
            "xPts",
            "Pts-xPts",
            "Past Opp Str",
            "Future Opp Str",
            "Next 5 Opp Str",
        ]
    ]
    .sort_values("xPts/g", ascending=False)
    .reset_index(drop=True)
)

display_df["team"] = display_df["team"].map(short_name_map)
display_df = display_df.rename(columns={"team": "Team"})
display_df.insert(0, "Rank", display_df.index + 1)


# Diverging colormaps.
# high_is_good: red -> white -> green (high values look good).
# low_is_good:  green -> white -> red (low values look good).
high_is_good = LinearSegmentedColormap.from_list(
    "hig", ["#c0392b", "#ffffff", "#27ae60"]
)
low_is_good = LinearSegmentedColormap.from_list(
    "lig", ["#27ae60", "#ffffff", "#c0392b"]
)


def diverging_gradient(styler, cols, cmap):
    """Apply a diverging gradient with the white point pinned at:
       - 0 if the column crosses zero
       - the column median otherwise
    Builds a per-column colormap so white lands at the correct fractional
    position and min/max always reach the full red/green endpoints."""
    for c in cols:
        s = styler.data[c]
        vmin = float(s.min())
        vmax = float(s.max())
        if s.min() < 0 < s.max():
            center = 0.0
        else:
            center = float(s.median())
        white_pos = (center - vmin) / (vmax - vmin)
        col_cmap = LinearSegmentedColormap.from_list(
            f"div_{c}",
            [(0.0, cmap(0.0)), (white_pos, "#ffffff"), (1.0, cmap(1.0))],
        )
        styler = styler.background_gradient(
            subset=[c], cmap=col_cmap, vmin=vmin, vmax=vmax
        )
    return styler


hig_cols = [
    "GF/g",
    "GD/g",
    "xGF/g",
    "xGD/g",
    "GD-xGD",
    "Pts",
    "xPts",
    "xPts/g",
    "Pts-xPts",
]
lig_cols = [
    "GA/g",
    "xGA/g",
    "Past Opp Str",
    "Future Opp Str",
    "Next 5 Opp Str",
]

strength_cols = ["Past Opp Str", "Future Opp Str", "Next 5 Opp Str"]
float_cols = [c for c in display_df.columns if c not in ("Rank", "Team", "GP")]
two_dp_cols = [c for c in float_cols if c not in strength_cols]

# pandas Styler resets prior subset formatters on each .format() call,
# so build one dict and apply formats in a single call.
fmt_map = {c: "{:.2f}" for c in two_dp_cols}
fmt_map.update({c: "{:.0f}" for c in strength_cols})
fmt_map["Rank"] = "{:d}"
fmt_map["GP"] = "{:d}"


def _text_color_for(bg_hex):
    r = int(bg_hex[1:3], 16)
    g = int(bg_hex[3:5], 16)
    b = int(bg_hex[5:7], 16)
    luminance = (0.299 * r + 0.587 * g + 0.114 * b) / 255
    return "white" if luminance < 0.6 else "black"


def style_team_col(s):
    out = []
    for v in s:
        bg = short_name_color_map.get(v)
        if bg is None:
            out.append("")
        else:
            out.append(
                f"background-color: {bg}; color: {_text_color_for(bg)};"
                " font-weight: bold;"
            )
    return out


last_game_date = elo_games_df.query("status == 'FullTime'").date_time_utc.max()
caption = (
    f"NWSL 2026 team performances through "
    f"{pd.to_datetime(last_game_date).strftime('%b %d, %Y')}"
)

(
    display_df.style.format(fmt_map)
    .pipe(diverging_gradient, hig_cols, high_is_good)
    .pipe(diverging_gradient, lig_cols, low_is_good)
    .apply(style_team_col, subset=["Team"])
    .set_caption(caption)
    .set_table_styles(
        [
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "1.1em"),
                    ("font-weight", "600"),
                    ("text-align", "left"),
                    ("padding", "6px 0 10px 0"),
                ],
            }
        ]
    )
    .hide(axis="index")
)

/var/folders/42/c1xq4jg10213hqzzpdz5xkjm0000gn/T/ipykernel_74589/2719238205.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


Rank,Team,GP,GF/g,GA/g,GD/g,xGF/g,xGA/g,xGD/g,GD-xGD,xPts/g,Pts,xPts,Pts-xPts,Past Opp Str,Future Opp Str,Next 5 Opp Str
1,San Diego,20,1.20,1.05,0.15,1.61,0.88,0.73,-11.61,1.87,33.00,37.40,-4.40,107,104,111
2,North Carolina,19,1.79,1.53,0.26,1.88,1.32,0.56,-5.62,1.72,30.00,32.70,-2.70,103,110,96
3,Gotham FC,20,1.25,0.65,0.60,1.59,1.08,0.51,1.85,1.72,43.00,34.35,8.65,104,104,109
4,Kansas City,20,1.50,1.40,0.10,1.77,1.29,0.48,-7.50,1.63,30.00,32.64,-2.64,105,106,104
5,Washington,20,1.45,0.85,0.60,1.50,1.12,0.38,4.38,1.62,37.00,32.50,4.50,110,89,79
6,Utah Royals,20,1.70,1.15,0.55,1.62,1.32,0.31,4.88,1.61,36.00,32.21,3.79,104,109,101
7,Angel City,18,1.50,1.11,0.39,1.52,1.31,0.21,3.23,1.49,25.00,26.90,-1.90,106,114,113
8,Orlando,20,1.20,1.55,-0.35,1.41,1.30,0.12,-9.40,1.47,23.00,29.37,-6.37,102,112,113
9,Seattle,19,1.21,1.32,-0.11,1.42,1.44,-0.02,-1.55,1.39,27.00,26.45,0.55,108,99,102
10,Denver,19,1.32,1.26,0.05,1.37,1.52,-0.15,3.82,1.33,24.00,25.33,-1.33,111,97,96
